# 00_ssb_build_schemas_and_config – Engangs infrastruktur-oppsett

Kjøres **kun én gang**, ved aller første oppsett av pipelinen i en ny lakehouse.
Oppretter de fire schemaene (namespacene) resten av pipelinen forutsetter finnes,
og legger inn en tom `ssb_config`-tabell med riktig struktur klar til å fylles
(via `02_ssb_config_admin`).

Trygt å kjøre flere ganger – `CREATE SCHEMA IF NOT EXISTS` gjør ingenting hvis
schemaet allerede finnes, og `ssb_config` overskrives kun hvis den er tom fra før
(kjør ikke denne på nytt hvis `ssb_config` allerede har data du vil beholde).

---

In [ ]:
#############################
# Bygg schemas
#############################
# Oppretter de fire "mappene" (schemaene) i lakehouset som resten av
# pipelinen sorterer tabellene sine inn i: pipeline (driftstabeller som
# ssb_config og ssb_load_queue), kodeverk (kommune-/regionkoder), ssb
# (de ferdige statistikktabellene) og fhi (reservert for fremtidig bruk).

spark.sql("CREATE SCHEMA IF NOT EXISTS statbank_staging.pipeline")
spark.sql("CREATE SCHEMA IF NOT EXISTS statbank_staging.kodeverk")
spark.sql("CREATE SCHEMA IF NOT EXISTS statbank_staging.ssb")
spark.sql("CREATE SCHEMA IF NOT EXISTS statbank_staging.fhi")

In [ ]:
############################
# Bygg ssb_config
############################
# Oppretter selve driftstabellen ssb_config – tom, men med riktig struktur.
# Dette er tabellen som senere sier HVILKE SSB-tabeller pipelinen skal følge
# med på, og holder styr på når hver av dem sist ble lastet ned og
# standardisert. Fylles med faktiske tabeller via 02_ssb_config_admin.

from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

SSB_CONFIG_SCHEMA = StructType([
    StructField("table_id",                  StringType(),  False),
    StructField("table_name",                StringType(),  True),
    StructField("frequency",                 StringType(),  True),
    StructField("category",                  StringType(),  True),
    StructField("lookback_periods",          IntegerType(), True),
    StructField("priority",                  StringType(),  True),
    StructField("last_downloaded_timestamp", StringType(),  True),
    StructField("last_loaded_timestamp",     StringType(),  True),
])

empty_df = spark.createDataFrame([], schema=SSB_CONFIG_SCHEMA)
empty_df.write.format("delta").mode("overwrite").saveAsTable("statbank_staging.pipeline.ssb_config")

print("✅ ssb_config opprettet (tom)")